# Sentiment Analysis on Amazon Product Reviews
### VortexTech AI/ML Internship — Week 4 (Capstone Project)

## Objective
Classify Amazon product reviews as **positive** or **negative** based on their
text — a binary text classification (NLP) problem.

## Dataset
**Source:** Amazon Reviews 2023 (Kaggle) — real, recent consumer reviews with
star ratings (1–5) and review text.

**Labels:** The dataset gives star ratings, not ready-made sentiment labels, so
we derive them:
- 4–5 stars → **Positive**
- 1–2 stars → **Negative**
- 3 stars → **Excluded** (ambiguous, would add label noise)

## Pipeline
1. Load & inspect data
2. Clean text
3. TF-IDF feature extraction
4. Train & compare models (Logistic Regression, Naive Bayes)
5. Evaluate (accuracy, F1, confusion matrix, error analysis)
6. Test on custom sentences
7. Summary & limitations

## Author
Sarim — VortexTech AI/ML Internship, Week 4

## Milestone 1: Load and balance the dataset

In [1]:
import pandas as pd
import numpy as np
import re
import ipywidgets as widgets
from IPython.display import display, HTML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# Load the dataset
df = pd.read_csv('../data/raw/Amazon_reviews_2023.csv')

print("Dataset shape:", df.shape)

print("Columns:", df.columns.tolist())
print(df.head())

print(df.info())

# check for missing values
print("Missing values per column:\n", df.isnull().sum())

Dataset shape: (701528, 10)
Columns: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']
   rating                                      title  \
0       5  Such a lovely scent but not overpowering.   
1       4     Works great but smells a little weird.   
2       5                                       Yes!   
3       1                          Synthetic feeling   
4       5                                         A+   

                                                text images        asin  \
0  This spray is really nice. It smells really go...     []  B00YQ6X8EO   
1  This product does what I need it to do, I just...     []  B081TJ8YS3   
2                          Smells good, feels great!     []  B07PNNCSP9   
3                                     Felt synthetic     []  B09JS339BZ   
4                                            Love it     []  B08BZ63GMJ   

  parent_asin                       user_id          

In [2]:
# Check missing values specifically in the columns we'll use
print("Missing 'text':", df['text'].isnull().sum())
print("Missing 'rating':", df['rating'].isnull().sum())

# Check the rating distribution — how many reviews per star rating?
print("\nRating distribution:\n", df['rating'].value_counts().sort_index())

Missing 'text': 212
Missing 'rating': 0

Rating distribution:
 rating
1    102080
2     43034
3     56307
4     79381
5    420726
Name: count, dtype: int64


## Label Derivation & Class Balancing

The raw dataset provides star ratings (1–5), not ready-made sentiment labels,
so we derive them ourselves:

- **4–5 stars → Positive**
- **1–2 stars → Negative**
- **3 stars → Excluded** (genuinely mixed/ambiguous sentiment; keeping them
  would add label noise to a binary classification task)

**Handling missing text:** 212 rows (~0.03% of the dataset) have no review
text. This is negligible, so we simply drop them rather than attempt
imputation.

**Handling class imbalance:** the raw rating distribution is heavily skewed
toward positive reviews (~60% are 5-star alone), giving roughly a 3.4:1
positive-to-negative ratio. Training on this directly risks a model that
achieves high accuracy simply by always predicting "positive," without
learning anything meaningful.

**Our approach:** since the dataset is large, we randomly sample an equal
number of positive and negative reviews (down-sampling the majority class)
to create a balanced subset. This is simpler to reason about than
alternatives like `class_weight='balanced'`, and keeps TF-IDF vectorization
and model training fast.

In [3]:
import numpy as np

df = df.dropna(subset=['text'])

# Keep only the columns we actually need
df = df[['text', 'rating']]

# Exclude 3-star reviews (ambiguous middle ground)
df = df[df['rating'] != 3]

# Derive binary sentiment label from rating
df['sentiment'] = df['rating'].apply(lambda r: 'positive' if r >= 4 else 'negative')

# Check class counts before balancing
print("Before balancing:\n", df['sentiment'].value_counts())

# Balance classes by sampling an equal number from each
SAMPLE_SIZE_PER_CLASS = 20000  # 20k positive + 20k negative = 40k total

positive_df = df[df['sentiment'] == 'positive'].sample(n=SAMPLE_SIZE_PER_CLASS, random_state=42)
negative_df = df[df['sentiment'] == 'negative'].sample(n=SAMPLE_SIZE_PER_CLASS, random_state=42)

# Combine and shuffle
df_balanced = pd.concat([positive_df, negative_df]).sample(frac=1, random_state=42).reset_index(drop=True)

print("\nAfter balancing:\n", df_balanced['sentiment'].value_counts())
print("\nFinal shape:", df_balanced.shape)
df_balanced.head()

Before balancing:
 sentiment
positive    499923
negative    145099
Name: count, dtype: int64

After balancing:
 sentiment
negative    20000
positive    20000
Name: count, dtype: int64

Final shape: (40000, 3)


,text,rating,sentiment
0,Great price but you get what you pay for. The...,1,negative
1,I haven't even used the matte yet but the regu...,5,positive
2,"these nails have a lovely design, however some...",2,negative
3,I like,5,positive
4,I followed the instructions perfectly. This pr...,1,negative


## Milestone 2: Text Cleaning

Before converting text into numeric features, we clean the raw review text.
The goal is to reduce noise while preserving the words that actually carry
sentiment.

**Steps:**
1. Lowercase all text — so "Great" and "great" are treated as the same word
2. Remove punctuation and special characters — reduces vocabulary size and
   noise (e.g. "good!!!" and "good" become identical)
3. Collapse extra whitespace left behind after cleaning

**On stopword removal:** we deliberately **do not** remove stopwords. Standard
stopword lists include negation words like "not," "no," and "never" — removing
them can flip a review's actual meaning (e.g. "not good" → "good" after naive
cleaning). Since negation is important signal for sentiment specifically, we
keep all words and let TF-IDF's built-in term-weighting handle very common,
low-information words instead.

In [4]:
import re

def clean_text(text):
    """
    Cleans a single review's text for sentiment analysis.

    Steps:
    1. Lowercase the text
    2. Remove punctuation/special characters (keep only letters and spaces)
    3. Collapse multiple spaces into one, strip leading/trailing whitespace

    Parameters
    ----------
    text : str
        The raw review text.

    Returns
    -------
    str
        The cleaned text.
    """
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)   # keep only letters and spaces
    text = re.sub(r'\s+', ' ', text).strip()  # collapse extra whitespace
    return text

# Apply cleaning to the balanced dataset
df_balanced['cleaned_text'] = df_balanced['text'].apply(clean_text)

# Compare a few original vs cleaned examples
df_balanced[['text', 'cleaned_text']].head(5)

,text,cleaned_text
0,Great price but you get what you pay for. The...,great price but you get what you pay for they ...
1,I haven't even used the matte yet but the regu...,i havent even used the matte yet but the regul...
2,"these nails have a lovely design, however some...",these nails have a lovely design however some ...
3,I like,i like
4,I followed the instructions perfectly. This pr...,i followed the instructions perfectly this pro...


## Milestone 3: Feature Extraction with TF-IDF

Machine learning models require numeric input, so we convert `cleaned_text`
into numeric features using **TF-IDF (Term Frequency–Inverse Document
Frequency)**.

TF-IDF scores each word in each review based on:
- **Term Frequency (TF):** how often the word appears in that specific review
- **Inverse Document Frequency (IDF):** how rare the word is across the whole
  dataset

A word scores highly only if it's frequent in a given review **and** rare
overall — this naturally down-weights very common, low-information words
(e.g. "the," "and") without needing an explicit stopword list, which aligns
with our earlier decision to keep negation words intact.

We cap the vocabulary at **5,000 features** (`max_features=5000`) to keep the
feature space manageable and avoid overfitting to rare, one-off words that
appear in only a single review.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize the vectorizer, capping vocabulary size at 5000 words
vectorizer = TfidfVectorizer(max_features=5000)

# Fit on cleaned text and transform into a TF-IDF feature matrix
X = vectorizer.fit_transform(df_balanced['cleaned_text'])

# Labels: convert 'positive'/'negative' strings into 1/0 for the model
y = df_balanced['sentiment'].map({'positive': 1, 'negative': 0})

print("Feature matrix shape:", X.shape)
print("Example vocabulary words:", vectorizer.get_feature_names_out()[:20])

Feature matrix shape: (40000, 5000)
Example vocabulary words: ['aa' 'ab' 'ability' 'able' 'about' 'above' 'abrasive' 'absolute'
 'absolutely' 'absorb' 'absorbed' 'absorbent' 'absorbing' 'absorbs'
 'accent' 'accept' 'acceptable' 'access' 'accessories' 'accessory']


## Milestone 4: Train/Test Split & Model Training

We split the TF-IDF feature matrix into training (80%) and test (20%) sets,
stratified by label to preserve class balance in both. The model never sees
the test set during training, which lets us evaluate honestly on unseen data.

We train **two** classifiers and compare them:
- **Logistic Regression** — learns a weight for each word (TF-IDF feature)
  indicating how strongly it pushes predictions toward positive or negative;
  applies a sigmoid function to output a probability
- **Multinomial Naive Bayes** — a probabilistic model commonly used as a
  strong, fast baseline for text classification

Comparing both gives us a more informed final choice than picking one model
arbitrarily.

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

# Split into train/test sets, stratified to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# Train Logistic Regression
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

# Train Multinomial Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

print("\nBoth models trained successfully.")

Train shape: (32000, 5000)
Test shape: (8000, 5000)

Both models trained successfully.


## Milestone 5: Evaluation

We evaluate both models on the held-out test set (never seen during training)
using:

- **Accuracy** — overall percentage of correct predictions
- **F1-score** — harmonic mean of precision and recall; balances false
  positives and false negatives into one number
- **Confusion matrix** — breaks down exactly which predictions were correct
  and which weren't, and in which direction the errors went

We then compare both models side by side to justify which one we carry
forward, and inspect a few misclassified test examples to understand
*where* and *why* the better model fails.

In [9]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# Generate predictions from both models
log_reg_preds = log_reg.predict(X_test)
nb_preds = nb_model.predict(X_test)

# Compare accuracy and F1-score side by side
print("Logistic Regression — Accuracy: {:.4f}, F1: {:.4f}".format(
    accuracy_score(y_test, log_reg_preds), f1_score(y_test, log_reg_preds)))

print("Multinomial Naive Bayes — Accuracy: {:.4f}, F1: {:.4f}".format(
    accuracy_score(y_test, nb_preds), f1_score(y_test, nb_preds)))

# Detailed report for the Logistic Regression model
print("\nLogistic Regression Classification Report:\n")
print(classification_report(y_test, log_reg_preds, target_names=['negative', 'positive']))

# Confusion matrix for Logistic Regression
cm = confusion_matrix(y_test, log_reg_preds)
print("Confusion Matrix (Logistic Regression):\n", cm)

Logistic Regression — Accuracy: 0.9015, F1: 0.9007
Multinomial Naive Bayes — Accuracy: 0.8845, F1: 0.8835

Logistic Regression Classification Report:

              precision    recall  f1-score   support

    negative       0.89      0.91      0.90      4000
    positive       0.91      0.89      0.90      4000

    accuracy                           0.90      8000
   macro avg       0.90      0.90      0.90      8000
weighted avg       0.90      0.90      0.90      8000

Confusion Matrix (Logistic Regression):
 [[3640  360]
 [ 428 3572]]


### Error Analysis

Aggregate metrics tell us *how much* the model is wrong, but not *why*.
Here we pull actual misclassified reviews from the test set to look for
patterns — e.g. sarcasm, mixed sentiment, negation, or very short reviews
with little signal.

In [23]:
import numpy as np

# Get the original test-set text back (X_test is numeric, so we need indices)
test_indices = y_test.index

# Build a comparison dataframe: original text, true label, predicted label
error_df = pd.DataFrame({
    'text': df_balanced.loc[test_indices, 'text'].values,
    'true_label': y_test.values,
    'predicted_label': log_reg_preds
})

# Filter to only the misclassified rows
misclassified = error_df[error_df['true_label'] != error_df['predicted_label']]

print(f"Total misclassified: {len(misclassified)} out of {len(error_df)}")

# Show a sample of misclassified reviews
misclassified.sample(10, random_state=42)

Total misclassified: 788 out of 8000


,text,true_label,predicted_label
5938,The hair was thick and nice but after 2 weeks ...,1,0
439,I also received two soufflés and no smoothie. ...,0,1
2224,Super cute but the little hoops that keep it t...,1,0
2107,Mask no moisturizes the skin!After using it my...,0,1
2503,When I opened this there was no inner seal and...,1,0
2263,Nice material and color assortment.<br />Butto...,1,0
5665,Leaves my hair very smooth so my comb does not...,1,0
3513,I just used this and had an instant difference...,1,0
3132,"Smells great, but that doesn’t last long. Rin...",0,1
1501,Much darker than expected for fair: light-medi...,0,1


## Milestone 6: Testing on Custom Sentences

We test the trained Logistic Regression model on three original sentences,
each designed to probe a specific challenge rather than an easy, obvious case:

1. **"The quality of these glasses is up to my expectations."** — subtle,
   low-signal sentiment with no strong positive/negative words
2. **"This toy is fun to play with but doesn't come cheap."** — positive
   clause followed by a negative one (the "but" pattern found in our error
   analysis)
3. **"This kitchen set is quite expensive but it justifies in terms of
   quality."** — the mirror case: negative clause followed by a positive one

Each sentence must go through the **same cleaning and vectorization steps**
used during training — the model only understands input in that exact
format.

In [10]:
# Our custom test sentences
custom_sentences = [
    "The quality of these glasses is up to my expectations.",
    "This toy is fun to play with but doesn't come cheap.",
    "This kitchen set is quite expensive but it justifies in terms of quality."
]

# Apply the SAME cleaning function used on the training data
cleaned_custom = [clean_text(sentence) for sentence in custom_sentences]

# Transform using the SAME fitted vectorizer (do NOT re-fit — use .transform, not .fit_transform)
custom_features = vectorizer.transform(cleaned_custom)

# Predict using the Logistic Regression model
custom_preds = log_reg.predict(custom_features)
custom_probs = log_reg.predict_proba(custom_features)

# Display results
for sentence, cleaned, pred, prob in zip(custom_sentences, cleaned_custom, custom_preds, custom_probs):
    label = "Positive" if pred == 1 else "Negative"
    confidence = prob[pred]
    print(f"Sentence: {sentence}")
    print(f"Cleaned:  {cleaned}")
    print(f"Prediction: {label} (confidence: {confidence:.2%})\n")

Sentence: The quality of these glasses is up to my expectations.
Cleaned:  the quality of these glasses is up to my expectations
Prediction: Positive (confidence: 57.42%)

Sentence: This toy is fun to play with but doesn't come cheap.
Cleaned:  this toy is fun to play with but doesnt come cheap
Prediction: Negative (confidence: 84.82%)

Sentence: This kitchen set is quite expensive but it justifies in terms of quality.
Cleaned:  this kitchen set is quite expensive but it justifies in terms of quality
Prediction: Positive (confidence: 71.29%)



## Milestone 7: Summary & Limitations

### Pipeline
- Converted ratings into binary sentiment labels (Positive: 4–5, Negative: 1–2; excluded 3-star reviews).
- Balanced the dataset with 20,000 reviews per class.
- Preprocessed text while preserving stopwords and negation words.
- Extracted features using TF-IDF (5,000 features).
- Trained and evaluated Logistic Regression and Multinomial Naive Bayes models.

### Results

| Model | Accuracy | F1-Score |
|-------|---------:|---------:|
| Logistic Regression | **90.15%** | **0.9007** |
| Multinomial Naive Bayes | 88.45% | 0.8835 |

- Logistic Regression achieved the best overall performance.
- Strong recall for both classes, with slightly better detection of negative reviews.

### Key Findings
- Most misclassifications occurred in reviews with mixed or contrasting sentiments.
- The model relied primarily on informative keywords rather than sentence structure.
- Predictions on ambiguous reviews reflected appropriate uncertainty.

### Limitations
- TF-IDF ignores word order and contextual relationships.
- Complex linguistic patterns such as sarcasm and contrast are difficult to capture.
- Context-aware models (e.g., BERT or other transformer-based models) could improve performance.

## Interactive Sentiment Checker

The cell below provides a simple interactive widget to test the trained
model on any sentence you type, in addition to the three predefined
examples above. This reuses the same cleaning and TF-IDF pipeline used
throughout the notebook.

**Known issue:** in some notebook environments, clicking "Predict" can
render the same result multiple times (a widget display/event-handler
quirk tied to how the notebook re-registers callbacks on cell re-runs,
common in Jupyter/VS Code). This does not affect the underlying model —
each duplicate shows the identical prediction and confidence score, so the
result itself is consistent and correct. This is a cosmetic limitation of
the interactive widget, not a bug in the classification pipeline.

In [11]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Close any previous instances of these widgets if the cell is re-run
try:
    text_input.close()
    predict_button.close()
    output.close()
except NameError:
    pass  # first run, nothing to close yet

# Create the widgets
text_input = widgets.Text(
    placeholder='Type a review sentence here...',
    description='Review:',
    layout=widgets.Layout(width='600px')
)
predict_button = widgets.Button(description='Predict Sentiment', button_style='primary')
# Prevent duplicate callbacks when re-running the cell (clear internal handler list if present)
if hasattr(predict_button, '_click_handlers'):
    try:
        predict_button._click_handlers.callbacks.clear()
    except Exception:
        pass
output = widgets.Output()

def on_predict_click(b):
    sentence = text_input.value.strip()
    output.clear_output(wait=True)
    with output:
        # Guard: ensure the trained model exists in the kernel
        if 'log_reg' not in globals() or 'vectorizer' not in globals():
            print("Model or vectorizer not found. Please run the training cells before using this widget.")
            return
        if not sentence:
            print("Please enter a sentence first.")
            return

        # Same pipeline as Milestone 6: clean -> transform -> predict
        cleaned = clean_text(sentence)
        features = vectorizer.transform([cleaned])
        pred = log_reg.predict(features)[0]
        prob = log_reg.predict_proba(features)[0]

        label = "Positive" if pred == 1 else "Negative"
        confidence = prob[pred]
        color = "green" if pred == 1 else "crimson"
        emoji = "😊" if pred == 1 else "😞"

        display(HTML(f"""
        <div style="padding:10px; border-left:4px solid {color}; font-family:sans-serif;">
            <b>Prediction:</b> <span style="color:{color}; font-size:16px;">{emoji} {label}</span><br>
            <b>Confidence:</b> {confidence:.1%}
        </div>
        """))

predict_button.on_click(on_predict_click)
display(text_input, predict_button, output)

Text(value='', description='Review:', layout=Layout(width='600px'), placeholder='Type a review sentence here..…

Button(button_style='primary', description='Predict Sentiment', style=ButtonStyle())

Output()